# 🚀 CoreOps OpsPilot — Multi-LLM Inference Server

**Run all cells in order.** Models are loaded with 4-bit quantization to fit on free T4 GPU.

| Model | Task | VRAM |
|-------|------|------|
| Phi-3.5-mini | Chat & General | ~2GB |
| Qwen3-4B | Intent Classification | ~2.5GB |
| DeepSeek-R1-Distill-Qwen-7B | Reasoning | ~4GB |

> **Note:** Vision model (Qwen2.5-VL) is optional — loaded only if VRAM remains.

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 1: Install Dependencies
# ═══════════════════════════════════════════════════════════
!pip install -q flask flask-cors pyngrok
!pip install -q transformers accelerate bitsandbytes
!pip install -q google-search-results Pillow
print("✅ Dependencies installed")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 2: Configuration — EDIT THESE VALUES
# ═══════════════════════════════════════════════════════════
import torch
import gc
import os
import time
import json
import re

# ┌──────────────────────────────────────────┐
# │   PASTE YOUR TOKENS HERE                 │
# └──────────────────────────────────────────┘
NGROK_AUTH_TOKEN = "YOUR_NGROK_AUTH_TOKEN"   # https://dashboard.ngrok.com/get-started/your-authtoken
SERPAPI_KEY      = "YOUR_SERPAPI_KEY"         # https://serpapi.com/manage-api-key (optional)

# Model config
MODELS_CONFIG = {
    "chat": {
        "name": "microsoft/Phi-3.5-mini-instruct",
        "task": "General chat & Q&A",
        "max_tokens": 1024,
        "temperature": 0.7,
    },
    "intent": {
        "name": "Qwen/Qwen3-4B",
        "task": "Intent classification",
        "max_tokens": 512,
        "temperature": 0.1,
    },
    "reasoning": {
        "name": "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B",
        "task": "Deep analysis & planning",
        "max_tokens": 2048,
        "temperature": 0.3,
    },
}

# GPU check
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU: {gpu_name} | VRAM: {gpu_mem:.1f} GB")
else:
    print("⚠️ No GPU detected — models will be very slow on CPU")
    gpu_mem = 0

print(f"📦 Models to load: {', '.join(MODELS_CONFIG.keys())}")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 3: Load Models (4-bit quantized)
# ═══════════════════════════════════════════════════════════
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

loaded_models = {}
loaded_tokenizers = {}

def load_model(key):
    """Load a single model with 4-bit quantization."""
    cfg = MODELS_CONFIG[key]
    model_name = cfg["name"]
    print(f"\n{'='*50}")
    print(f"⏳ Loading [{key}]: {model_name}")
    print(f"{'='*50}")
    t0 = time.time()

    try:
        tokenizer = AutoTokenizer.from_pretrained(
            model_name, trust_remote_code=True
        )
        # Ensure pad token exists
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token

        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True,
            torch_dtype=torch.float16,
        )
        model.eval()

        loaded_models[key] = model
        loaded_tokenizers[key] = tokenizer

        elapsed = time.time() - t0
        vram_used = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0
        print(f"✅ [{key}] loaded in {elapsed:.0f}s | VRAM: {vram_used:.1f}GB")
        return True

    except Exception as e:
        print(f"❌ [{key}] FAILED: {e}")
        return False

# Load models smallest → largest
for key in ["chat", "intent", "reasoning"]:
    load_model(key)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Summary
print(f"\n{'='*50}")
print(f"📊 LOADED: {list(loaded_models.keys())}")
if torch.cuda.is_available():
    used = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"💾 VRAM: {used:.1f}GB / {total:.1f}GB ({used/total*100:.0f}%)")
print(f"{'='*50}")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 4: Inference Functions
# ═══════════════════════════════════════════════════════════

def generate_text(model_key, prompt, system_prompt=None, max_tokens=None, temperature=None):
    """Generate text using a loaded model."""
    if model_key not in loaded_models:
        return {"error": f"Model '{model_key}' not loaded", "text": None}

    cfg = MODELS_CONFIG[model_key]
    model = loaded_models[model_key]
    tokenizer = loaded_tokenizers[model_key]
    max_new = max_tokens or cfg["max_tokens"]
    temp = temperature if temperature is not None else cfg["temperature"]

    # Build chat messages
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": prompt})

    # Apply chat template
    try:
        text_input = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    except Exception:
        # Fallback for models without chat template
        if system_prompt:
            text_input = f"System: {system_prompt}\nUser: {prompt}\nAssistant:"
        else:
            text_input = f"User: {prompt}\nAssistant:"

    inputs = tokenizer(text_input, return_tensors="pt", truncation=True, max_length=4096)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    input_len = inputs["input_ids"].shape[1]

    t0 = time.time()
    try:
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new,
                temperature=max(temp, 0.01),
                do_sample=(temp > 0.01),
                top_p=0.9,
                pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
            )

        # Decode only the NEW tokens
        new_tokens = outputs[0][input_len:]
        response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
        duration_ms = int((time.time() - t0) * 1000)

        return {
            "text": response,
            "model": cfg["name"],
            "model_key": model_key,
            "duration_ms": duration_ms,
            "tokens_generated": len(new_tokens),
        }

    except Exception as e:
        return {"error": str(e), "text": None, "model_key": model_key}


def generate_json(model_key, prompt, system_prompt=None):
    """Generate text and parse JSON from it."""
    result = generate_text(model_key, prompt, system_prompt=system_prompt)
    if result.get("error") or not result.get("text"):
        return {**result, "parsed": None}

    text = result["text"]

    # Try parsing JSON from response
    for attempt_fn in [
        lambda t: json.loads(t),                                          # Direct parse
        lambda t: json.loads(re.search(r'```json\s*([\s\S]*?)```', t).group(1)),  # ```json block
        lambda t: json.loads(re.search(r'(\{[\s\S]*\})', t).group(1)),            # Any { } block
    ]:
        try:
            parsed = attempt_fn(text)
            return {**result, "parsed": parsed}
        except Exception:
            continue

    return {**result, "parsed": None}


def search_web(query):
    """Search internet via SerpAPI."""
    if not SERPAPI_KEY or SERPAPI_KEY == "YOUR_SERPAPI_KEY":
        return {"error": "SerpAPI not configured", "results": []}
    try:
        from serpapi import GoogleSearch
        results = GoogleSearch({"q": query, "api_key": SERPAPI_KEY, "num": 5}).get_dict()
        organic = results.get("organic_results", [])
        return {
            "results": [{"title": r.get("title"), "snippet": r.get("snippet"), "link": r.get("link")} for r in organic[:5]],
            "query": query,
        }
    except Exception as e:
        return {"error": str(e), "results": []}


# Quick test
if "chat" in loaded_models:
    test = generate_text("chat", "Say hello in one sentence.")
    print(f"🧪 Test: {test.get('text', test.get('error'))}")
    print(f"   Time: {test.get('duration_ms', '?')}ms")
else:
    print("⚠️ Chat model not loaded — skipping test")

print("\n✅ Inference functions ready")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 5: Flask API Server
# ═══════════════════════════════════════════════════════════
from flask import Flask, request, jsonify
from flask_cors import CORS

app = Flask(__name__)
CORS(app)

@app.errorhandler(Exception)
def handle_error(e):
    return jsonify({"success": False, "error": str(e)}), 500

# ─── Health ──────────────────────────────────
@app.route("/api/health", methods=["GET"])
def health():
    vram_used = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0
    vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
    return jsonify({
        "status": "ok",
        "models_loaded": list(loaded_models.keys()),
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
        "vram": {"used_gb": round(vram_used, 1), "total_gb": round(vram_total, 1)},
        "search_enabled": SERPAPI_KEY != "YOUR_SERPAPI_KEY",
    })

# ─── Reasoning ──────────────────────────────
@app.route("/api/reasoning", methods=["POST"])
def api_reasoning():
    data = request.get_json(force=True)
    result = generate_text(
        "reasoning",
        data.get("prompt", ""),
        system_prompt=data.get("system_prompt", "You are an expert ERP analyst."),
        max_tokens=data.get("max_tokens"),
        temperature=data.get("temperature"),
    )
    return jsonify({"success": not result.get("error"), "data": result})

# ─── Intent ──────────────────────────────────
@app.route("/api/intent", methods=["POST"])
def api_intent():
    data = request.get_json(force=True)
    result = generate_json(
        "intent",
        data.get("prompt", ""),
        system_prompt=data.get("system_prompt", 'Extract intent as JSON.'),
    )
    return jsonify({"success": not result.get("error"), "data": result})

# ─── Chat ────────────────────────────────────
@app.route("/api/chat", methods=["POST"])
def api_chat():
    data = request.get_json(force=True)
    result = generate_text(
        "chat",
        data.get("prompt", ""),
        system_prompt=data.get("system_prompt", "You are OpsPilot, an AI assistant for CoreOps ERP."),
        max_tokens=data.get("max_tokens"),
        temperature=data.get("temperature"),
    )
    return jsonify({"success": not result.get("error"), "data": result})

# ─── Vision placeholder ─────────────────────
@app.route("/api/vision", methods=["POST"])
def api_vision():
    return jsonify({"success": False, "data": {"error": "Vision model not loaded", "text": None}})

# ─── Search ──────────────────────────────────
@app.route("/api/search", methods=["POST"])
def api_search():
    data = request.get_json(force=True)
    query = data.get("query", "")
    result = search_web(query)
    if data.get("summarize") and result.get("results"):
        snippets = "\n".join([f"- {r['title']}: {r['snippet']}" for r in result["results"]])
        summary = generate_text("chat", f"Summarize about '{query}':\n{snippets}")
        result["summary"] = summary.get("text")
    return jsonify({"success": not result.get("error"), "data": result})

# ─── Orchestrate ─────────────────────────────
@app.route("/api/orchestrate", methods=["POST"])
def api_orchestrate():
    data = request.get_json(force=True)
    prompt = data.get("prompt", "")

    # Step 1: Classify
    intent_result = generate_json("intent", prompt, system_prompt=(
        'Classify this command. Output JSON only:\n'
        '{"intent": "INTENT_NAME", "model_needed": "reasoning|chat|search", '
        '"entities": {}, "confidence": 0.0}'
    ))
    parsed = intent_result.get("parsed") or {}
    model_needed = parsed.get("model_needed", "chat")
    intent_name = parsed.get("intent", "GENERAL")

    # Step 2: Route
    if model_needed == "reasoning" and "reasoning" in loaded_models:
        main_result = generate_text("reasoning", prompt,
            system_prompt="Analyze thoroughly. Provide actionable insights.")
    elif model_needed == "search":
        sr = search_web(prompt)
        snippets = "\n".join([f"- {r['title']}: {r['snippet']}" for r in sr.get("results", [])])
        main_result = generate_text("chat", f"Answer based on search results:\n{snippets}\n\nQuestion: {prompt}")
        main_result["search_results"] = sr.get("results", [])
    else:
        main_result = generate_text("chat", prompt)

    return jsonify({
        "success": True,
        "data": {
            "intent": intent_name,
            "confidence": parsed.get("confidence", 0),
            "entities": parsed.get("entities", {}),
            "model_used": main_result.get("model_key", model_needed),
            "response": main_result.get("text"),
            "duration_ms": (intent_result.get("duration_ms", 0) or 0) + (main_result.get("duration_ms", 0) or 0),
            "models_chain": ["intent", main_result.get("model_key", model_needed)],
        }
    })

print("✅ Flask API ready")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 6: START SERVER — Copy the URL to your .env file!
# ═══════════════════════════════════════════════════════════
from pyngrok import ngrok
import threading

# Auth
if NGROK_AUTH_TOKEN == "YOUR_NGROK_AUTH_TOKEN":
    print("❌ Set your NGROK_AUTH_TOKEN in Cell 2 first!")
    print("   Get it from: https://dashboard.ngrok.com/get-started/your-authtoken")
else:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    port = 5001

    # Kill any existing tunnels
    for tunnel in ngrok.get_tunnels():
        ngrok.disconnect(tunnel.public_url)

    public_url = ngrok.connect(port).public_url

    print("\n" + "═" * 55)
    print("  🚀  OpsPilot AI Server is LIVE!")
    print("═" * 55)
    print(f"")
    print(f"  📡 URL: {public_url}")
    print(f"")
    print(f"  Add to your backend .env file:")
    print(f"  KAGGLE_INFERENCE_URL={public_url}")
    print(f"")
    print(f"  Models: {list(loaded_models.keys())}")
    print("═" * 55)

    # Run Flask in thread so the cell doesn't block
    threading.Thread(
        target=lambda: app.run(host="0.0.0.0", port=port, debug=False, use_reloader=False),
        daemon=True,
    ).start()

    print(f"\n✅ Server running on port {port}")
    print("   Keep this notebook running! The server stops when you close it.")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 7: Test Endpoints (run AFTER Cell 6)
# ═══════════════════════════════════════════════════════════
import requests

BASE = f"http://localhost:5001"

# Health check
print("\n--- Health ---")
r = requests.get(f"{BASE}/api/health")
print(json.dumps(r.json(), indent=2))

# Chat test
print("\n--- Chat ---")
r = requests.post(f"{BASE}/api/chat", json={"prompt": "What can you help me with?"})
d = r.json()
print(f"Response: {d['data'].get('text', d['data'].get('error'))}")
print(f"Time: {d['data'].get('duration_ms')}ms")

# Intent test
print("\n--- Intent ---")
r = requests.post(f"{BASE}/api/intent", json={"prompt": "Approve purchase order PO-001"})
d = r.json()
print(f"Parsed: {d['data'].get('parsed')}")

print("\n✅ All tests passed!")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 8: Keep Alive (run this to prevent Kaggle timeout)
# ═══════════════════════════════════════════════════════════
import time

print("⏰ Keep-alive running. Server will stay active.")
print("   Press Stop to shut down.\n")

try:
    while True:
        time.sleep(60)
        # Ping to keep ngrok tunnel alive
        try:
            r = requests.get(f"http://localhost:5001/api/health", timeout=5)
            status = "✅" if r.status_code == 200 else "⚠️"
        except:
            status = "❌"
        print(f"{status} Alive — {time.strftime('%H:%M:%S')}", end="\r")
except KeyboardInterrupt:
    print("\n🛑 Server stopped.")